# "Bare" 1-table Test: Goal: show what is known/assumed already

Note: The prediction does not fit the result.

In [1]:
import random

# ==========================================
# 1. SETUP: Secret 128-Bit Key & Pearson Table
# ==========================================
# Secret salt array (4 x u32 = 128 bits of entropy)
SECRET_SALT = [0xDEADBEEF, 0xCAFEBABE, 0x12345678, 0x9ABCDEF0]

# Generate secret 256-byte Pearson permutation table (T) from the salt
random.seed(sum(SECRET_SALT))
T = list(range(256))
random.shuffle(T)

# Inverse table used by the sender to compute ciphertext bytes
T_inv = [0] * 256
for i in range(256):
    T_inv[T[i]] = i

# ==========================================
# 2. SENDER: Encode Plaintext into Ciphertext
# ==========================================
def encode(plaintext: bytes, initial_state: int = 0) -> list[int]:
    ciphertext = []
    current_state = initial_state
    for m in plaintext:
        # We need T[current_state ^ c] == m
        # Therefore: c = current_state ^ T_inv[m]
        c = current_state ^ T_inv[m]
        ciphertext.append(c)
        current_state = m  # Next state becomes the current ASCII byte
    return ciphertext

# ==========================================
# 3. RECEIVER: Decode Ciphertext using Secret Table T
# ==========================================
def decode_legit(ciphertext: list[int], initial_state: int = 0) -> bytes:
    plaintext = []
    current_state = initial_state
    for c in ciphertext:
        next_state = T[current_state ^ c]
        plaintext.append(next_state)
        current_state = next_state
    return bytes(plaintext)


# ==========================================
# DEMONSTRATION
# ==========================================

# Secret message sent over the wire
FULL_MESSAGE = b"Dear Commander, the rendezvous point is at Sector 7 at midnight."
INITIAL_STATE = 0x42  # E.g., timestamp or validation start byte

# Sender encodes the message
ciphertext = encode(FULL_MESSAGE, INITIAL_STATE)

print("--- 1. LEGITIMATE TRANSMISSION ---")
print(f"Original Text : {FULL_MESSAGE.decode()}")
print(f"Ciphertext    : {bytes(ciphertext)[:30]}... (64 bytes total)\n")

# Verify legit receiver can read it
decrypted_legit = decode_legit(ciphertext, INITIAL_STATE)
assert decrypted_legit == FULL_MESSAGE


# ==========================================
# 4. ATTACKER: Decrypt without Secret Key
# ==========================================
print("--- 2. ATTACKER RECONSTRUCTION ---")

# Suppose the attacker knows/guesses the opening 16 bytes: "Dear Commander, "
KNOWN_HEADER = b"Dear Commander, "
SECRET_TAIL   = FULL_MESSAGE[len(KNOWN_HEADER):] # "the rendezvous point..."

# Attacker initializes an empty table of 256 entries (all set to None)
attacker_table = [None] * 256
attacker_state = INITIAL_STATE

# Step A: Attacker uses KNOWN text to learn table entries
for i in range(len(KNOWN_HEADER)):
    p = KNOWN_HEADER[i]
    c = ciphertext[i]

    # Mathematical identity: Table[State XOR Ciphertext] = Plaintext
    table_index = attacker_state ^ c
    attacker_table[table_index] = p

    attacker_state = p  # Update state to the known ASCII byte

learned_entries = sum(1 for x in attacker_table if x is not None)
print(f"Attacker learned {learned_entries} table entries from header '{KNOWN_HEADER.decode()}'\n")

# Step B: Attacker decrypts the UNKNOWN rest of the message using learned table
recovered_bytes = []
for i in range(len(KNOWN_HEADER), len(ciphertext)):
    c = ciphertext[i]
    table_index = attacker_state ^ c

    decrypted_byte = attacker_table[table_index]

    if decrypted_byte is not None:
        recovered_bytes.append(decrypted_byte)
        attacker_state = decrypted_byte  # State stays synchronized!
    else:
        recovered_bytes.append(ord('?')) # Unknown mapping
        # State desyncs if byte is completely unknown

decrypted_tail = bytes(recovered_bytes).decode('ascii', errors='replace')

print(f"Target Secret Tail : {SECRET_TAIL.decode()}")
print(f"Attacker Decrypted  : {decrypted_tail}")

"""
--- 1. LEGITIMATE TRANSMISSION ---
Original Text : Dear Commander, the rendezvous point is at Sector 7 at midnight.
Ciphertext    : b'lqe\x9eYq\x0c\xdb\xd9m\xc9\x8dQ\x9ax\x07K\x06]N\xdfG\xcd\x8dQ&\x069y\x8c'... (64 bytes total)

--- 2. ATTACKER RECONSTRUCTION ---
Attacker learned 11 table entries from header 'Dear Commander, '

Target Secret Tail : the rendezvous point is at Sector 7 at midnight.
Attacker Decrypted  : ????rende??????????????????????????????????e????
\noutput: partially decoded\n--- 1. LEGITIMATE TRANSMISSION ---\nOriginal
Text : Dear Commander, the rendezvous point is at Sector 7 at midnight.
\nCiphertext
: b'lqe\x9eYq\x0cÛÙmÉ\x8dQ\x9ax\x07K\x06]NßGÍ\x8dQ&\x069y\x8c'...
(64 bytes total)\n\n--- 2. ATTACKER RECONSTRUCTION ---\nAttacker learned
11 table entries from header 'Dear Commander, '\n\nTarget Secret Tail :
the rendezvous point is at Sector 7 at midnight.\nAttacker Decrypted  :
????rende??????????????????????????????????e????\n
"""

--- 1. LEGITIMATE TRANSMISSION ---
Original Text : Dear Commander, the rendezvous point is at Sector 7 at midnight.
Ciphertext    : b'lqe\x9eYq\x0c\xdb\xd9m\xc9\x8dQ\x9ax\x07K\x06]N\xdfG\xcd\x8dQ&\x069y\x8c'... (64 bytes total)

--- 2. ATTACKER RECONSTRUCTION ---
Attacker learned 11 table entries from header 'Dear Commander, '

Target Secret Tail : the rendezvous point is at Sector 7 at midnight.
Attacker Decrypted  : ????rende??????????????????????????????????e????


"\noutput: partially decoded\n--- 1. LEGITIMATE TRANSMISSION ---\nOriginal Text : Dear Commander, the rendezvous point is at Sector 7 at midnight.\nCiphertext    : b'lqe\x9eYq\x0cÛÙmÉ\x8dQ\x9ax\x07K\x06]NßGÍ\x8dQ&\x069y\x8c'... (64 bytes total)\n\n--- 2. ATTACKER RECONSTRUCTION ---\nAttacker learned 11 table entries from header 'Dear Commander, '\n\nTarget Secret Tail : the rendezvous point is at Sector 7 at midnight.\nAttacker Decrypted  : ????rende??????????????????????????????????e????\n"

In [6]:
# Standard 256-byte Pearson permutation table (RFC 1413 / standard Pearson)
STANDARD_PEARSON_TABLE = [
    98, 6, 85, 150, 36, 23, 112, 164, 135, 207, 169, 5, 26, 64, 165, 219,
    61, 20, 68, 89, 130, 63, 52, 102, 24, 229, 132, 245, 80, 216, 195, 115,
    90, 168, 156, 203, 177, 120, 2, 190, 188, 7, 100, 185, 174, 243, 162, 10,
    237, 18, 253, 225, 8, 208, 172, 244, 255, 126, 101, 79, 145, 235, 228, 121,
    123, 251, 67, 250, 161, 0, 107, 97, 241, 111, 181, 82, 249, 33, 69, 55,
    59, 153, 29, 9, 213, 167, 84, 93, 30, 46, 94, 75, 151, 114, 73, 222,
    197, 96, 210, 45, 16, 227, 248, 202, 51, 152, 252, 125, 81, 206, 215, 186,
    39, 158, 178, 187, 131, 136, 1, 49, 50, 17, 141, 91, 47, 129, 60, 99,
    154, 35, 86, 171, 105, 34, 38, 200, 147, 58, 77, 118, 173, 246, 76, 254,
    133, 232, 196, 144, 198, 124, 53, 4, 108, 74, 223, 234, 134, 230, 157, 139,
    189, 205, 199, 128, 176, 19, 211, 236, 127, 192, 231, 70, 233, 88, 146, 44,
    136, 201, 166, 137, 160, 106, 224, 116, 83, 212, 247, 226, 221, 21, 214, 179,
    113, 103, 15, 11, 140, 183, 117, 238, 11, 220, 175, 25, 143, 14, 56, 110,
    78, 48, 191, 138, 155, 62, 12, 194, 170, 71, 148, 104, 159, 122, 109, 42,
    217, 180, 193, 149, 22, 218, 87, 27, 92, 142, 43, 163, 65, 147, 41, 182,
    184, 32, 28, 119, 129, 37, 31, 242, 171, 40, 239, 13, 204, 127, 209, 57
]

# Build the public inverse table T_inv
T_INV = [0] * 256
for idx, val in enumerate(STANDARD_PEARSON_TABLE):
    T_INV[val] = idx


# ================================================================
# 1. SECRET KEY: A 16-Byte Salt Array (e.g., derived from 4 x u32)
# ================================================================
SECRET_SALT_ARRAY = [
    0xAB, 0x12, 0xCD, 0x34, 0x56, 0xef, 0x78, 0x90,
    0xAA, 0xBB, 0xCC, 0xDD, 0xEE, 0xFF, 0x11, 0x22
]
SALT_LEN = len(SECRET_SALT_ARRAY)


# ================================================================
# 2. SENDER: Encode using cumulative Salted Pearson Hash
# ================================================================
def encode_salted(plaintext: bytes, initial_state: int = 0) -> list[int]:
    ciphertext = []
    current_state = initial_state

    for i, m in enumerate(plaintext):
        salt = SECRET_SALT_ARRAY[i % SALT_LEN]

        # Formula: m = T[current_state ^ c ^ salt]
        # So: T_inv[m] = current_state ^ c ^ salt
        # Therefore: c = current_state ^ salt ^ T_inv[m]
        c = current_state ^ salt ^ T_INV[m]

        ciphertext.append(c)
        current_state = m # Next running state is target ASCII character

    return ciphertext


# ================================================================
# 3. RECEIVER: Decode using cumulative Salted Pearson Hash
# ================================================================
def decode_salted(ciphertext: list[int], initial_state: int = 0) -> bytes:
    plaintext = []
    current_state = initial_state

    for i, c in enumerate(ciphertext):
        salt = SECRET_SALT_ARRAY[i % SALT_LEN]

        # Formula: next_state = T[current_state ^ c ^ salt]
        next_state = STANDARD_PEARSON_TABLE[current_state ^ c ^ salt]
        plaintext.append(next_state)
        current_state = next_state

    return bytes(plaintext)


# ================================================================
# DEMONSTRATION
# ================================================================

MESSAGE = b"Dear Commander, the rendezvous point is at Sector 7 at midnight."
INIT_STATE = 0x00

# Sender encodes the message
ciphertext = encode_salted(MESSAGE, INIT_STATE)

# Receiver verifies it works
assert decode_salted(ciphertext, INIT_STATE) == MESSAGE
print("--- TRANSMISSION SUCCESSFUL ---")
print(f"Message: {MESSAGE.decode()}\n")


# ================================================================
# 4. ATTACKER: Extract the Secret Salt Array from Known Plaintext
# ================================================================
print("--- ATTACKER EXTRACTS SECRET SALT ARRAY ---")

KNOWN_PREFIX = b"Dear Commander, " # 16 bytes of known text

recovered_salt_array = [None] * SALT_LEN
attacker_state = INIT_STATE

for i in range(len(KNOWN_PREFIX)):
    m = KNOWN_PREFIX[i]
    c = ciphertext[i]

    # Attacker formula: Salt = T_inv[m] ^ previous_state ^ ciphertext
    extracted_salt_byte = T_INV[m] ^ attacker_state ^ c

    salt_index = i % SALT_LEN
    recovered_salt_array[salt_index] = extracted_salt_byte

    attacker_state = m # Attacker tracks the state

print(f"Actual Secret Salts   : {SECRET_SALT_ARRAY}")
print(f"Attacker Extracted    : {recovered_salt_array}\n")

# Verify the attacker recovered the exact secret key
assert recovered_salt_array == SECRET_SALT_ARRAY
print("SUCCESS: The attacker completely extracted the secret salt array!")

# Now the attacker can decrypt any past or future messages using the extracted salt array
decrypted_by_attacker = decode_salted(ciphertext, INIT_STATE)
print(f"Attacker Decrypted Message: {decrypted_by_attacker.decode()}")


"""


output: 100% decoded
--- TRANSMISSION SUCCESSFUL ---
Message: Dear Commander, the rendezvous point is at Sector 7 at midnight.

--- ATTACKER EXTRACTS SECRET SALT ARRAY ---
Actual Secret Salts   : [171, 18, 205, 52, 86, 239, 120, 144, 170, 187, 204, 221, 238, 255, 17, 34]
Attacker Extracted    : [171, 18, 205, 52, 86, 239, 120, 144, 170, 187, 204, 221, 238, 255, 17, 34]

SUCCESS: The attacker completely extracted the secret salt array!
Attacker Decrypted Message: Dear Commander, the rendezvous point is at Sector 7 at midnight.

"""

'''
--- TRANSMISSION SUCCESSFUL ---
Message: Dear Commander, the rendezvous point is at Sector 7 at midnight.

--- ATTACKER EXTRACTS SECRET SALT ARRAY ---
Actual Secret Salts   : [171, 18, 205, 52, 86, 239, 120, 144, 170, 187, 204, 221, 238, 255, 17, 34]
Attacker Extracted    : [171, 18, 205, 52, 86, 239, 120, 144, 170, 187, 204, 221, 238, 255, 17, 34]

SUCCESS: The attacker completely extracted the secret salt array!
Attacker Decrypted Message: Dear Commander, the rendezvous point is at Sector 7 at midnight.
\noutput: 100% decoded\n--- TRANSMISSION SUCCESSFUL ---\nMessage:
Dear Commander, the rendezvous point is at Sector 7 at midnight.\n\n---
 ATTACKER EXTRACTS SECRET SALT ARRAY ---\nActual Secret Salts   :
 [171, 18, 205, 52, 86, 239, 120, 144, 170, 187, 204, 221, 238, 255, 17, 34]
 \nAttacker Extracted    :
 [171, 18, 205, 52, 86, 239, 120, 144, 170, 187, 204, 221, 238, 255, 17, 34]
 \n\nSUCCESS: The attacker completely extracted the secret salt array!
 \nAttacker Decrypted Message: Dear Commander,
 the rendezvous point is at Sector 7 at midnight.\n\n\n
'''

--- TRANSMISSION SUCCESSFUL ---
Message: Dear Commander, the rendezvous point is at Sector 7 at midnight.

--- ATTACKER EXTRACTS SECRET SALT ARRAY ---
Actual Secret Salts   : [171, 18, 205, 52, 86, 239, 120, 144, 170, 187, 204, 221, 238, 255, 17, 34]
Attacker Extracted    : [171, 18, 205, 52, 86, 239, 120, 144, 170, 187, 204, 221, 238, 255, 17, 34]

SUCCESS: The attacker completely extracted the secret salt array!
Attacker Decrypted Message: Dear Commander, the rendezvous point is at Sector 7 at midnight.


'\noutput: 100% decoded\n--- TRANSMISSION SUCCESSFUL ---\nMessage: Dear Commander, the rendezvous point is at Sector 7 at midnight.\n\n--- ATTACKER EXTRACTS SECRET SALT ARRAY ---\nActual Secret Salts   : [171, 18, 205, 52, 86, 239, 120, 144, 170, 187, 204, 221, 238, 255, 17, 34]\nAttacker Extracted    : [171, 18, 205, 52, 86, 239, 120, 144, 170, 187, 204, 221, 238, 255, 17, 34]\n\nSUCCESS: The attacker completely extracted the secret salt array!\nAttacker Decrypted Message: Dear Commander, the rendezvous point is at Sector 7 at midnight.\n\n\n'

# pingpong test

In [7]:
import math
from collections import Counter

# ==========================================
# 1. PINGPONG CRYPTO REFERENCE IMPLEMENTATION
# ==========================================

PAYLOAD_SIZE = 54

DEFAULT_SALTS = [
    0x0F1E2D3C, 0x4B5A6978, 0x8796A5B4, 0xC3D2E1F0,
    0x12345678, 0x9ABCDEF0, 0xFEDCBA98, 0x76543210,
    0x00112233, 0x44556677, 0x8899AABB, 0xCCDDEEFF,
    0xA1B2C3D4, 0xE5F60718, 0x29384756, 0x61728394
]

class PingPongEngine:
    def __init__(self, salts: list[int], nonce: int):
        seed_1, seed_2 = self._derive_seeds(salts, nonce)
        self.t1, self.inv1 = self._scramble_table(seed_1)
        self.t2, self.inv2 = self._scramble_table(seed_2)

    def _derive_seeds(self, salts: list[int], nonce: int) -> tuple[int, int]:
        seed_1 = 0xCBF29CE484222325
        for s in salts:
            val = (s ^ nonce) & 0xFFFFFFFF
            seed_1 ^= val
            seed_1 = (seed_1 * 0x00000100000001B3) & 0xFFFFFFFFFFFFFFFF
        seed_2 = seed_1 ^ 0xA5A5A5A5A5A5A5A5
        return seed_1, seed_2

    def _scramble_table(self, seed: int) -> tuple[list[int], list[int]]:
        table = list(range(256))
        prng = seed
        for idx in range(255, 0, -1):
            prng = (prng * 6364136223846793005 + 1442695040888963407) & 0xFFFFFFFFFFFFFFFF
            rand_val = (prng >> 32) & 0xFFFFFFFF
            j = rand_val % (idx + 1)
            table[idx], table[j] = table[j], table[idx]

        inv = [0] * 256
        for k in range(256):
            inv[table[k]] = k
        return table, inv

    def _swap_and_update(self, t: list[int], inv: list[int], i: int, j: int):
        if i == j:
            return
        val_i, val_j = t[i], t[j]
        t[i], t[j] = val_j, val_i
        inv[val_j] = i
        inv[val_i] = j

    def _mutate_t2(self, x: int):
        i1 = x & 0xFF
        j1 = (i1 + 13) & 0xFF
        self._swap_and_update(self.t2, self.inv2, i1, j1)

        i2 = (i1 * 7 + 3) & 0xFF
        j2 = (i2 + 101) & 0xFF
        self._swap_and_update(self.t2, self.inv2, i2, j2)

    def _mutate_t1(self, c: int):
        i1 = c & 0xFF
        j1 = (i1 + 17) & 0xFF
        self._swap_and_update(self.t1, self.inv1, i1, j1)

        i2 = (i1 * 11 + 5) & 0xFF
        j2 = (i2 + 67) & 0xFF
        self._swap_and_update(self.t1, self.inv1, i2, j2)

    def encrypt_payload(self, plaintext: bytes) -> bytes:
        ciphertext = bytearray(PAYLOAD_SIZE)
        state = 0
        for i in range(PAYLOAD_SIZE):
            m = plaintext[i]
            idx1 = (state ^ m) & 0xFF
            x = self.t1[idx1]
            c = self.t2[x]
            ciphertext[i] = c
            state = m

            self._mutate_t2(x)
            self._mutate_t1(c)

        return bytes(ciphertext)


# ==========================================
# 2. ANALYSIS & ENTROPY METRICS
# ==========================================

def calculate_shannon_entropy(data: bytes) -> float:
    """Calculates Shannon Entropy in bits per byte (Max = log2(54) = ~5.75 for 54 bytes)."""
    if not data:
        return 0.0
    counts = Counter(data)
    length = len(data)
    return -sum((cnt / length) * math.log2(cnt / length) for cnt in counts.values())

def count_repeated_adjacent_bytes(data: bytes) -> int:
    """Counts instances where c[i] == c[i+1]."""
    return sum(1 for i in range(len(data) - 1) if data[i] == data[i+1])

def max_run_length(data: bytes) -> int:
    """Finds the longest sequence of identical adjacent bytes."""
    if not data:
        return 0
    max_run, current_run = 1, 1
    for i in range(1, len(data)):
        if data[i] == data[i-1]:
            current_run += 1
            max_run = max(max_run, current_run)
        else:
            current_run = 1
    return max_run

def run_test_case(name: str, plaintext: bytes, nonce: int = 0x12345678):
    engine = PingPongEngine(DEFAULT_SALTS, nonce)
    ciphertext = engine.encrypt_payload(plaintext)
    entropy = calculate_shannon_entropy(ciphertext)
    adj_repeats = count_repeated_adjacent_bytes(ciphertext)
    longest_run = max_run_length(ciphertext)
    unique_bytes = len(set(ciphertext))

    print(f"=== TEST: {name} ===")
    print(f"Plaintext (54B) : {plaintext[:30]}...")
    print(f"Ciphertext Hex  : {ciphertext.hex()[:60]}...")
    print(f"Shannon Entropy : {entropy:.4f} / 5.7549 (Max for 54 bytes)")
    print(f"Unique Bytes    : {unique_bytes} / 54")
    print(f"Adjacent Repeats: {adj_repeats}")
    print(f"Max Run Length  : {longest_run}\n")

# ==========================================
# 3. RUNNING SCENARIOS
# ==========================================

if __name__ == "__main__":
    # Scenario A: All 'a's
    run_test_case("Repeated Single Character ('a' * 54)", b"a" * 54)

    # Scenario B: Repeated 'Hi'
    run_test_case("Repeated Pattern ('Hi' * 27)", b"Hi" * 27)

    # Scenario C: Natural Language (Control Group)
    run_test_case("Natural English Text", b"Dear Commander, the rendezvous point is Sector 7 at 12".ljust(54, b"\x00"))

"""
output: EPIC FAIL!!!! 'Mark it, Nuncle'

=== TEST: Repeated Single Character ('a' * 54) ===
Plaintext (54B) : b'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaa'...
Ciphertext Hex  : 10f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4...
Shannon Entropy : 1.1143 / 5.7549 (Max for 54 bytes)
Unique Bytes    : 3 / 54
Adjacent Repeats: 0
Max Run Length  : 1

=== TEST: Repeated Pattern ('Hi' * 27) ===
Plaintext (54B) : b'HiHiHiHiHiHiHiHiHiHiHiHiHiHiHi'...
Ciphertext Hex  : 7586f786f786f786f786f786f786f786f786f786f786f786f786f786f786...
Shannon Entropy : 1.1143 / 5.7549 (Max for 54 bytes)
Unique Bytes    : 3 / 54
Adjacent Repeats: 0
Max Run Length  : 1

=== TEST: Natural English Text ===
Plaintext (54B) : b'Dear Commander, the rendezvous'...
Ciphertext Hex  : 77863413ebd69805f4cf1a182368ad003fb827f3adea4bb6700bcfca08c6...
Shannon Entropy : 5.4216 / 5.7549 (Max for 54 bytes)
Unique Bytes    : 45 / 54
Adjacent Repeats: 0
Max Run Length  : 1

"""

=== TEST: Repeated Single Character ('a' * 54) ===
Plaintext (54B) : b'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaa'...
Ciphertext Hex  : 10f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4...
Shannon Entropy : 1.1143 / 5.7549 (Max for 54 bytes)
Unique Bytes    : 3 / 54
Adjacent Repeats: 0
Max Run Length  : 1

=== TEST: Repeated Pattern ('Hi' * 27) ===
Plaintext (54B) : b'HiHiHiHiHiHiHiHiHiHiHiHiHiHiHi'...
Ciphertext Hex  : 7586f786f786f786f786f786f786f786f786f786f786f786f786f786f786...
Shannon Entropy : 1.1143 / 5.7549 (Max for 54 bytes)
Unique Bytes    : 3 / 54
Adjacent Repeats: 0
Max Run Length  : 1

=== TEST: Natural English Text ===
Plaintext (54B) : b'Dear Commander, the rendezvous'...
Ciphertext Hex  : 77863413ebd69805f4cf1a182368ad003fb827f3adea4bb6700bcfca08c6...
Shannon Entropy : 5.4216 / 5.7549 (Max for 54 bytes)
Unique Bytes    : 45 / 54
Adjacent Repeats: 0
Max Run Length  : 1



"\noutput: EPIC FAIL!!!! 'Mark it, Nuncle'\n\n=== TEST: Repeated Single Character ('a' * 54) ===\nPlaintext (54B) : b'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaa'...\nCiphertext Hex  : 10f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4b5f4...\nShannon Entropy : 1.1143 / 5.7549 (Max for 54 bytes)\nUnique Bytes    : 3 / 54\nAdjacent Repeats: 0\nMax Run Length  : 1\n\n=== TEST: Repeated Pattern ('Hi' * 27) ===\nPlaintext (54B) : b'HiHiHiHiHiHiHiHiHiHiHiHiHiHiHi'...\nCiphertext Hex  : 7586f786f786f786f786f786f786f786f786f786f786f786f786f786f786...\nShannon Entropy : 1.1143 / 5.7549 (Max for 54 bytes)\nUnique Bytes    : 3 / 54\nAdjacent Repeats: 0\nMax Run Length  : 1\n\n=== TEST: Natural English Text ===\nPlaintext (54B) : b'Dear Commander, the rendezvous'...\nCiphertext Hex  : 77863413ebd69805f4cf1a182368ad003fb827f3adea4bb6700bcfca08c6...\nShannon Entropy : 5.4216 / 5.7549 (Max for 54 bytes)\nUnique Bytes    : 45 / 54\nAdjacent Repeats: 0\nMax Run Length  : 1\n\n"

# Counter-Injection Test 1

In [8]:
# ==========================================
# COMPARISON: Original vs Counter-Injected
# ==========================================

def encrypt_original(plaintext: bytes, T1: list[int], T2: list[int]) -> bytes:
    ciphertext = bytearray(len(plaintext))
    state = 0
    for i in range(len(plaintext)):
        m = plaintext[i]
        idx1 = (state ^ m) & 0xFF  # NO COUNTER
        x = T1[idx1]
        c = T2[x]
        ciphertext[i] = c
        state = m                  # Plaintext feedback
    return bytes(ciphertext)

def encrypt_hardened(plaintext: bytes, T1: list[int], T2: list[int], key_byte: int) -> bytes:
    ciphertext = bytearray(len(plaintext))
    state = 0
    for i in range(len(plaintext)):
        m = plaintext[i]
        # 1. COUNTER + CIPHERTEXT-FEEDBACK INJECTION
        idx1 = (state ^ m ^ i) & 0xFF
        x = T1[idx1]
        c = T2[x]
        ciphertext[i] = c
        state = c                      # Ciphertext feedback (CBC style)

        # 2. DYNAMIC SWAP OFFSETS (No more static 13, 7, 3, 101)
        offset_1 = (i + key_byte) | 1
        offset_2 = (x ^ key_byte) | 1

        # Dynamic mutation of T2
        j1 = (x + offset_1) & 0xFF
        T2[x], T2[j1] = T2[j1], T2[x]

        i2 = (x * 5 + offset_2) & 0xFF
        j2 = (i2 + offset_1) & 0xFF
        T2[i2], T2[j2] = T2[j2], T2[i2]

    return bytes(ciphertext)

# --- RUNNING THE TEST ---
import random
random.seed(42)
T1_init = list(range(256)); random.shuffle(T1_init)
T2_init = list(range(256)); random.shuffle(T2_init)

msg = b"a" * 54

out_orig = encrypt_original(msg, list(T1_init), list(T2_init))
out_hard = encrypt_hardened(msg, list(T1_init), list(T2_init), key_byte=0x7A)

print("--- INPUT: 54 Repeated 'a' Bytes ---")
print(f"Original Code Output Hex : {out_orig.hex()[:40]}...")
print(f"Original Unique Bytes    : {len(set(out_orig))} / 54")
print()
print(f"Hardened Code Output Hex : {out_hard.hex()[:40]}...")
print(f"Hardened Unique Bytes    : {len(set(out_hard))} / 54")


"""

--- INPUT: 54 Repeated 'a' Bytes ---
Original Code Output Hex : 2930303030303030303030303030303030303030...
Original Unique Bytes    : 2 / 54

Hardened Code Output Hex : 29b7ee74e2994f37f962de8b792ac0f9bcc637ab...
Hardened Unique Bytes    : 49 / 54
"""

--- INPUT: 54 Repeated 'a' Bytes ---
Original Code Output Hex : 2930303030303030303030303030303030303030...
Original Unique Bytes    : 2 / 54

Hardened Code Output Hex : 29b7ee74e2994f37f962de8b792ac0f9bcc637ab...
Hardened Unique Bytes    : 49 / 54


"\n\n--- INPUT: 54 Repeated 'a' Bytes ---\nOriginal Code Output Hex : 2930303030303030303030303030303030303030...\nOriginal Unique Bytes    : 2 / 54\n\nHardened Code Output Hex : 29b7ee74e2994f37f962de8b792ac0f9bcc637ab...\nHardened Unique Bytes    : 49 / 54\n"

# Avalanche Efect Test 1

In [5]:
#!/usr/bin/env python3
"""
# Avalanche Effect Test

PingPong Cryptanalysis & Verification Sandbox
Demonstrates:
  1. Vulnerability of static Pearson tables to Known-Plaintext Attacks.
  2. Entropy & Avalanche effect of Hardened PingPong v2 under repetitive patterns.
"""

import math
import os
import random
from collections import Counter

# --- Configuration & Constants ---
PAYLOAD_SIZE = 54
DEFAULT_SALTS = [
    0x0F1E2D3C, 0x4B5A6978, 0x8796A5B4, 0xC3D2E1F0,
    0x12345678, 0x9ABCDEF0, 0xFEDCBA98, 0x76543210,
    0x00112233, 0x44556677, 0x8899AABB, 0xCCDDEEFF,
    0xA1B2C3D4, 0xE5F60718, 0x29384756, 0x61728394
]

# --- 1. Hardened PingPong v2 Reference Implementation ---
class HardenedPingPongEngine:
    def __init__(self, salts: list[int], nonce: int):
        self.salts = salts
        seed_1, seed_2 = self._derive_seeds(salts, nonce)
        self.t1, self.inv1 = self._scramble_table(seed_1)
        self.t2, self.inv2 = self._scramble_table(seed_2)

    def _derive_seeds(self, salts: list[int], nonce: int) -> tuple[int, int]:
        seed_1 = 0xCBF29CE484222325
        for s in salts:
            val = (s ^ nonce) & 0xFFFFFFFF
            seed_1 ^= val
            seed_1 = (seed_1 * 0x00000100000001B3) & 0xFFFFFFFFFFFFFFFF
        seed_2 = seed_1 ^ 0xA5A5A5A5A5A5A5A5
        return seed_1, seed_2

    def _scramble_table(self, seed: int) -> tuple[list[int], list[int]]:
        table = list(range(256))
        prng = seed
        for idx in range(255, 0, -1):
            prng = (prng * 6364136223846793005 + 1442695040888963407) & 0xFFFFFFFFFFFFFFFF
            rand_val = (prng >> 32) & 0xFFFFFFFF
            j = rand_val % (idx + 1)
            table[idx], table[j] = table[j], table[idx]
        inv = [0] * 256
        for k in range(256):
            inv[table[k]] = k
        return table, inv

    def _swap_and_update(self, t: list[int], inv: list[int], i: int, j: int):
        if i == j:
            return
        vi, vj = t[i], t[j]
        t[i], t[j] = vj, vi
        inv[vj], inv[vi] = i, j

    def _mutate_t2(self, x: int, salt_val: int):
        i1 = x & 0xFF
        offset = (salt_val | 1) & 0xFF
        j1 = (i1 + offset) & 0xFF
        self._swap_and_update(self.t2, self.inv2, i1, j1)

        i2 = (i1 * 5 + offset) & 0xFF
        j2 = (i2 + offset) & 0xFF
        self._swap_and_update(self.t2, self.inv2, i2, j2)

    def _mutate_t1(self, c: int, salt_val: int):
        i1 = c & 0xFF
        offset = (salt_val | 1) & 0xFF
        j1 = (i1 + offset) & 0xFF
        self._swap_and_update(self.t1, self.inv1, i1, j1)

        i2 = (i1 * 3 + offset) & 0xFF
        j2 = (i2 + offset) & 0xFF
        self._swap_and_update(self.t1, self.inv1, i2, j2)

    def encrypt_payload(self, plaintext: bytes) -> bytes:
        ciphertext = bytearray(PAYLOAD_SIZE)
        state = 0
        for i in range(PAYLOAD_SIZE):
            m = plaintext[i]
            idx1 = (state ^ m ^ i) & 0xFF
            x = self.t1[idx1]
            c = self.t2[x]
            ciphertext[i] = c
            state = c

            salt_val = self.salts[i & 15] & 0xFF
            self._mutate_t2(x, salt_val)
            self._mutate_t1(c, salt_val)
        return bytes(ciphertext)

    def decrypt_payload(self, ciphertext: bytes) -> bytes:
        plaintext = bytearray(PAYLOAD_SIZE)
        state = 0
        for i in range(PAYLOAD_SIZE):
            c = ciphertext[i]
            x = self.inv2[c]
            raw = self.inv1[x]
            m = (state ^ raw ^ i) & 0xFF
            plaintext[i] = m
            state = c

            salt_val = self.salts[i & 15] & 0xFF
            self._mutate_t2(x, salt_val)
            self._mutate_t1(c, salt_val)
        return bytes(plaintext)


# --- 2. Cryptanalytic Verification Metrics ---
def shannon_entropy(data: bytes) -> float:
    if not data:
        return 0.0
    counts = Counter(data)
    length = len(data)
    return -sum((cnt / length) * math.log2(cnt / length) for cnt in counts.values())

def hamming_distance(b1: bytes, b2: bytes) -> int:
    return sum(bin(x ^ y).count('1') for x, y in zip(b1, b2))

def run_cryptanalysis():
    print("==================================================")
    print("PINGPONG V2 CRYPTANALYSIS & ENTROPY EVALUATION")
    print("==================================================")

    nonce = 0xDEADBEEF
    engine = HardenedPingPongEngine(DEFAULT_SALTS, nonce)

    # Test Case 1: Repeated Inputs ("a" * 54)
    msg_a = b"a" * PAYLOAD_SIZE
    cipher_a = engine.encrypt_payload(msg_a)
    entropy_a = shannon_entropy(cipher_a)
    unique_a = len(set(cipher_a))

    print(f"\n[Test 1] Repetitive Plaintext ('a' * 54):")
    print(f"  Ciphertext (Hex) : {cipher_a.hex()[:60]}...")
    print(f"  Shannon Entropy  : {entropy_a:.4f} / 5.7549 (Theoretical max for 54 bytes)")
    print(f"  Unique Bytes     : {unique_a} / 54")

    # Roundtrip verification
    engine_dec = HardenedPingPongEngine(DEFAULT_SALTS, nonce)
    decrypted_a = engine_dec.decrypt_payload(cipher_a)
    assert decrypted_a == msg_a, "Decryption roundtrip failed!"
    print("  Decryption Status: PASSED (100% Match)")

    # Test Case 2: Avalanche Effect (Single-bit flip in plaintext)
    msg_b = bytearray(msg_a)
    msg_b[0] = ord('b')  # Flip 1 byte

    e1 = HardenedPingPongEngine(DEFAULT_SALTS, nonce)
    e2 = HardenedPingPongEngine(DEFAULT_SALTS, nonce)
    c1 = e1.encrypt_payload(msg_a)
    c2 = e2.encrypt_payload(msg_b)

    bit_diff = hamming_distance(c1, c2)
    bit_ratio = bit_diff / (PAYLOAD_SIZE * 8)

    print(f"\n[Test 2] Avalanche Effect (1 Byte Change in Input):")
    print(f"  Flipped Bits     : {bit_diff} / {PAYLOAD_SIZE * 8} bits")
    print(f"  Bit Diff Ratio   : {bit_ratio * 100.0:.2f}% (Target: ~50%)")

    assert entropy_a > 4.5, "Entropy check failed!"
    assert bit_ratio > 0.35, "Avalanche effect check failed!"
    print("\n[Result] All Cryptanalytic Assertions Passed Successfully.")

if __name__ == "__main__":
    run_cryptanalysis()


"""
==================================================
PINGPONG V2 CRYPTANALYSIS & ENTROPY EVALUATION
==================================================

[Test 1] Repetitive Plaintext ('a' * 54):
  Ciphertext (Hex) : 2e99e5fb0273d7467d7a88776f70fcc82fca618c1c8e837e20c3bcd55180...
  Shannon Entropy  : 5.6438 / 5.7549 (Theoretical max for 54 bytes)
  Unique Bytes     : 51 / 54
  Decryption Status: PASSED (100% Match)

[Test 2] Avalanche Effect (1 Byte Change in Input):
  Flipped Bits     : 195 / 432 bits
  Bit Diff Ratio   : 45.14% (Target: ~50%)

[Result] All Cryptanalytic Assertions Passed Successfully.

\n==================================================
\nPINGPONG V2 CRYPTANALYSIS & ENTROPY EVALUATION\n
==================================================
\n\n[Test 1] Repetitive Plaintext ('a' * 54):\n  Ciphertext
(Hex) : 2e99e5fb0273d7467d7a88776f70fcc82fca618c1c8e837e20c3bcd55180...
\n  Shannon Entropy  : 5.6438 / 5.7549 (Theoretical max for 54 bytes)\n
 Unique Bytes     : 51 / 54\n  Decryption Status: PASSED (100% Match)\n\n
 [Test 2] Avalanche Effect (1 Byte Change in Input):\n  Flipped Bits
 : 195 / 432 bits\n  Bit Diff Ratio   : 45.14% (Target: ~50%)\n\n[Result]
  All Cryptanalytic Assertions Passed Successfully.\n
"""

PINGPONG V2 CRYPTANALYSIS & ENTROPY EVALUATION

[Test 1] Repetitive Plaintext ('a' * 54):
  Ciphertext (Hex) : 2e99e5fb0273d7467d7a88776f70fcc82fca618c1c8e837e20c3bcd55180...
  Shannon Entropy  : 5.6438 / 5.7549 (Theoretical max for 54 bytes)
  Unique Bytes     : 51 / 54
  Decryption Status: PASSED (100% Match)

[Test 2] Avalanche Effect (1 Byte Change in Input):
  Flipped Bits     : 195 / 432 bits
  Bit Diff Ratio   : 45.14% (Target: ~50%)

[Result] All Cryptanalytic Assertions Passed Successfully.


"\n==================================================\nPINGPONG V2 CRYPTANALYSIS & ENTROPY EVALUATION\n==================================================\n\n[Test 1] Repetitive Plaintext ('a' * 54):\n  Ciphertext (Hex) : 2e99e5fb0273d7467d7a88776f70fcc82fca618c1c8e837e20c3bcd55180...\n  Shannon Entropy  : 5.6438 / 5.7549 (Theoretical max for 54 bytes)\n  Unique Bytes     : 51 / 54\n  Decryption Status: PASSED (100% Match)\n\n[Test 2] Avalanche Effect (1 Byte Change in Input):\n  Flipped Bits     : 195 / 432 bits\n  Bit Diff Ratio   : 45.14% (Target: ~50%)\n\n[Result] All Cryptanalytic Assertions Passed Successfully.\n"